In [2]:
# === SETUP: Run this first! ===
import os
import sys

# Change to project root and add to Python path
notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir)  # Goes up one level from 'notebooks/'
os.chdir(project_root)
sys.path.insert(0, project_root)

print(f"Project root: {project_root}")
print(f"tsnn module path: {os.path.join(project_root, 'tsnn')}")

Project root: /home/ubuntu
tsnn module path: /home/ubuntu/tsnn


In [3]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.linear_model import LassoCV, RidgeCV, LinearRegression
import torch
from torch.utils.data import Dataset, random_split
from torch.utils.data import DataLoader
import importlib
import sys
#sys.path.append('/Users/cyrilgarcia/notebooks/tsnn/')

import tsnn

from tsnn.generators import generators
from tsnn.benchmarks import benchmark_comparison, ml_benchmarks, torch_benchmarks
from tsnn import utils
import torch.nn.functional as F
import math
from typing import Optional
from tsnn.tstorch import transformers



plt.style.use('ggplot')

/opt/pytorch/lib/python3.12/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [4]:
from dataclasses import dataclass
from torch import nn


In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'mps')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")

Using device: cuda
GPU name: Tesla T4
Number of GPUs: 1


In [6]:
from typing import Dict

In [7]:
from tsnn.tstorch import models

In [8]:
from tsnn.tstorch.models import GlobalMLP, BiDimensionalMLP, OneDimensionalTransformer, CustomBiDimensionalTransformer
from sklearn.ensemble import HistGradientBoostingRegressor



# Summary

In [9]:
# In this notebook we will run the experiments to generate the figures for the paper.

# Test dataset

In [10]:
# We work with the following data.

In [11]:
# Global parameters
T_max = 5000
N1 = 10
F1 = 20
T1 = 5 # This parameter will be the n_rolling

print(T_max, N1, F1, T1)

5000 10 20 5


In [12]:
def generate_synthetic_datasets(
    num_time_steps: int = 3000,
    num_time_series: int = 10,
    num_features: int = 10,
    low_corr: float = 0.1,
    high_corr: float = 0.2,
    pct_zero_corr: float = 0.5,
) -> Dict[str, "generators.Generator"]:
    """
    Generates 5 synthetic multivariate time series datasets with different
    types of cross-series dependencies.

    Returns
    -------
    dict
        Keys: "d_lin", "d_cond", "d_shift", "d_cs", "d_cs_shift", "d_all"
        Values: generators.Generator objects (already with .train and .test)
    """
    dic_data = {}

    # Helper to avoid repeating the same 10 lines
    def make_gen(split_conditional=0.0,
                 split_shift=0.0,
                 split_seasonal=0.0,
                 split_cs=0.0,
                 split_cs_shift=0.0):
        gen = generators.Generator(num_time_steps, num_time_series, num_features)
        gen.generate_dataset(
            pct_zero_corr=pct_zero_corr,
            split_conditional=split_conditional,
            split_shift=split_shift,
            split_seasonal=split_seasonal,
            split_cs=split_cs,
            split_cs_shift=split_cs_shift,
            low_corr=low_corr,
            high_corr=high_corr,
        )
        return gen

    dic_data["d_lin"] = make_gen()

    # 1. Pure conditional (causal) dependence
    dic_data["d_cond"] = make_gen(split_conditional=1.0)

    # 2. Pure lagged (time-shifted) dependence
    dic_data["d_shift"] = make_gen(split_shift=1.0)

    # 3. Pure contemporaneous cross-sectional correlation
    dic_data["d_cs"] = make_gen(split_cs=1.0)

    # 4. Contemporaneous + lagged cross-series
    dic_data["d_cs_shift"] = make_gen(split_cs_shift=1.0)

    # 5. Equal mix of all four mechanisms
    dic_data["d_all"] = make_gen(
        split_conditional=0.2,
        split_shift=0.2,
        split_cs=0.2,
        split_cs_shift=0.2,
    )

    return dic_data

In [13]:
# list_low_corr = [0.01, 0.025, 0.05, 0.1]
# list_high_corr = [2*x for x in list_low_corr]

list_low_corr = [0.01, 0.03, 0.05, 0.1, 0.3, 0.5]
list_high_corr = list_low_corr

dic_data = {}

for i in range(len(list_low_corr)):
    name = "correl" + str(list_low_corr[i])
    dic_data[name] = generate_synthetic_datasets(num_time_steps=T_max, num_time_series=N1, num_features=F1, low_corr=list_low_corr[i], high_corr=list_high_corr[i])
    

In [14]:
dic_data.keys()

dict_keys(['correl0.01', 'correl0.03', 'correl0.05', 'correl0.1', 'correl0.3', 'correl0.5'])

In [15]:
effects = list(dic_data['correl0.1'].keys())
print(effects)

['d_lin', 'd_cond', 'd_shift', 'd_cs', 'd_cs_shift', 'd_all']


In [16]:
# We will fix the above dataset for now.

In [17]:
def causal_mask(b, h, q_idx, kv_idx):
    return q_idx >= kv_idx

def build_attention_mask(mask_fn, seq_len, device="cpu"):
    q_idx = torch.arange(seq_len, device=device)
    kv_idx = torch.arange(seq_len, device=device)
    b = torch.zeros(1, device=device)
    h = torch.zeros(1, device=device)
    mask_bool = mask_fn(b, h, q_idx[:, None], kv_idx[None, :])  # (seq_len, seq_len)
    return mask_bool

mask = causal_mask
mask = build_attention_mask(mask, T1, device=device)
def custom_mask_mod(b, h, q_idx, kv_idx):
    return mask[q_idx, kv_idx]

# List of models

In [18]:
# Let's list here all the models we wish to test on all the data.

In [19]:
def get_models():
    MLP_global = GlobalMLP(N1, F1, T1, dropout=0.2).to(device)

    MLP_2D = BiDimensionalMLP(N1, F1, T1, dropout=0.2).to(device)

    trans_1D_T4 = OneDimensionalTransformer(N1, F1, T1, mask=mask, attn_direction="T",  num_attn_layers=4,
                                            dropout=0.2, roll_y=True).to(device)
    #Note: using the MLP compression seems very bad..

    trans_2D_TCTC = models.CustomBiDimensionalTransformer(N1, F1, T1, mask=mask, layers='TCTC',
                                                                        dropout=0.05, d_model=128, dim_feedforward=128*4, sparsify=None, 
                                                                               roll_y=True).to(device)

    trans_2D_TCTCTCTC = models.CustomBiDimensionalTransformer(N1, F1, T1, mask=mask, layers='TCTCTCTC',
                                                                        dropout=0.05, d_model=128, dim_feedforward=128*4, sparsify=None, 
                                                                               roll_y=True).to(device)


    dic_models = {'MLP_global':MLP_global, 'MLP_2D':MLP_2D, "trans_1D_T4":trans_1D_T4, "trans_2D_TCTC":trans_2D_TCTC, "trans_2D_TCTCTCTC":trans_2D_TCTCTCTC}

    trans_2D_TCTC_rollfalse = models.CustomBiDimensionalTransformer(N1, F1, T1, mask=mask, layers='TCTC',
                                                                        dropout=0.05, d_model=128, dim_feedforward=128*4, sparsify=None, 
                                                                               roll_y=False).to(device)

    trans_2D_TCTCTCTC_rollfalse = models.CustomBiDimensionalTransformer(N1, F1, T1, mask=mask, layers='TCTCTCTC',
                                                                        dropout=0.05, d_model=128, dim_feedforward=128*4, sparsify=None, 
                                                                               roll_y=False).to(device)


    dic_models_rollfalse = {"trans_2D_TCTC":trans_2D_TCTC_rollfalse, "trans_2D_TCTCTCTC":trans_2D_TCTCTCTC_rollfalse}
    
    return dic_models, dic_models_rollfalse

In [20]:
# For each model we also need to specify the option we will use to fit them.

In [21]:
dic_data['correl0.1']

{'d_lin': <tsnn.generators.generators.Generator at 0x757b94653380>,
 'd_cond': <tsnn.generators.generators.Generator at 0x757b94653290>,
 'd_shift': <tsnn.generators.generators.Generator at 0x757b946535c0>,
 'd_cs': <tsnn.generators.generators.Generator at 0x757b94653800>,
 'd_cs_shift': <tsnn.generators.generators.Generator at 0x757b94653a10>,
 'd_all': <tsnn.generators.generators.Generator at 0x757b94653950>}

In [22]:
dic_data.keys()

dict_keys(['correl0.01', 'correl0.03', 'correl0.05', 'correl0.1', 'correl0.3', 'correl0.5'])

# Function to create table for an effect

In [23]:
# We give the function that creates for a given effect the table testing all models and all noise level.

In [33]:
def run_models(effect1):

    # Storage
    records_train = []
    records_test  = []

    for noise_level in dic_data.keys():
        
        z = dic_data[noise_level][effect1]
        z.get_dataloader(n_rolling=T1)

        dic_models, dic_models_rollfalse = get_models()


        lasso_full = ml_benchmarks.CustomBenchmarkRolling(LassoCV(alphas=np.linspace(start=1e-4, stop=1e-2, num=30), 
                                                                verbose=False, max_iter=2000, selection='random', n_jobs=5, cv=3))
        lasso_full.fit(z.train)
        boost_model = ml_benchmarks.CustomBenchmarkRolling(HistGradientBoostingRegressor(max_depth=6, max_iter=500, validation_fraction=0.3, n_iter_no_change=100))
        boost_model.fit(z.train)

        comp = benchmark_comparison.Comparator(models=[lasso_full, boost_model], model_names=['lasso_full', 'boosting'])
        corr_train = comp.correl(z, mode="train", return_values=True)
        corr_test  = comp.correl(z, mode="test",  return_values=True)

        # Save results
        records_train.append({
            "noise_level": noise_level,
            "model": 'lasso_full',
            "train_corr_optimal": corr_train.loc['lasso_full', "optimal"]
        })
        records_test.append({
            "noise_level": noise_level,
            "model": 'lasso_full',
            "test_corr_optimal": corr_test.loc['lasso_full', "optimal"]
        })

        records_train.append({
            "noise_level": noise_level,
            "model": 'boosting',
            "train_corr_optimal": corr_train.loc['boosting', "optimal"]
        })
        records_test.append({
            "noise_level": noise_level,
            "model": 'boosting',
            "test_corr_optimal": corr_test.loc['boosting', "optimal"]
        })
        


        for model_key in dic_models.keys():                   
            print(f"Running → {noise_level} | {model_key}")

            z = dic_data[noise_level][effect1]
            if model_key in ["trans_1D_T4", "trans_2D_TCTC", "trans_2D_TCTCTCTC"]:
                z.get_dataloader(n_rolling=T1, roll_y=True)
            else:
                z.get_dataloader(n_rolling=T1)

            if model_key in ["trans_2D_TCTC", "trans_2D_TCTCTCTC"]:
                lr=0.001/2
            else:
                lr=0.001

            epochs = 20
            if noise_level in ['correl0.01', 'correl0.03']:
                epochs = 40

            model = dic_models[model_key]

            # Model
            if noise_level == 'correl0.01' and model_key in ["trans_2D_TCTC", "trans_2D_TCTCTCTC"]:
                z.get_dataloader(n_rolling=T1)
                model = dic_models_rollfalse[model_key]
                epochs = 60
                lr=0.0001

            optimizer = torch.optim.Adam(model.parameters(), lr=lr)
            wrapper = torch_benchmarks.TorchWrapper(model, optimizer=optimizer, device=device)

            
            # Train
            wrapper.fit(z.train, test=z.test, epochs=epochs, plot=False)

            # Compare
            comp = benchmark_comparison.Comparator(models=[wrapper], model_names=["model1"])

            corr_train = comp.correl(z, mode="train", return_values=True)
            corr_test  = comp.correl(z, mode="test",  return_values=True)

            train_corr = corr_train.loc["model1", "optimal"]
            test_corr  = corr_test.loc["model1", "optimal"]

            # Save results
            records_train.append({
                "noise_level": noise_level,
                "model": model_key,
                "train_corr_optimal": train_corr
            })
            records_test.append({
                "noise_level": noise_level,
                "model": model_key,
                "test_corr_optimal": test_corr
            })

    # ————————————————————————————————————————
    # 3. Convert to nice DataFrames
    # ————————————————————————————————————————
    df_train = pd.DataFrame(records_train)
    df_test  = pd.DataFrame(records_test)

    # Pivot: rows = noise level, columns = effect type
    train_pivot = df_train.pivot(index="noise_level", columns="model", values="train_corr_optimal")
    test_pivot  = df_test.pivot(index="noise_level", columns="model", values="test_corr_optimal")

    col_order = ["lasso_full", "boosting"] + list(dic_models.keys())
    train_pivot = train_pivot[col_order]
    test_pivot  = test_pivot[col_order]

    return train_pivot, test_pivot

## Running on linear effect

In [34]:
dic_models, dic_models_rollfalse = get_models()

table_train_lin0, table_test_lin0 = run_models('d_lin')

Running → correl0.01 | MLP_global


100%|██████████| 40/40 [00:06<00:00,  6.03it/s]


Running → correl0.01 | MLP_2D


100%|██████████| 40/40 [00:07<00:00,  5.33it/s]


Running → correl0.01 | trans_1D_T4


...

100%|██████████| 20/20 [00:07<00:00,  2.80it/s]


Running → correl0.5 | trans_2D_TCTC


100%|██████████| 20/20 [00:11<00:00,  1.75it/s]


Running → correl0.5 | trans_2D_TCTCTCTC


100%|██████████| 20/20 [00:21<00:00,  1.07s/it]


In [35]:
display(table_train_lin0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,,,
correl0.01,-0.009,0.005,0.028,0.026,0.030,0.221,0.202
correl0.03,0.029,0.024,0.088,0.084,0.092,0.103,0.094
correl0.05,0.046,0.047,0.160,0.171,0.198,0.411,0.415
correl0.1,0.154,0.087,0.314,0.334,0.381,0.746,0.720
correl0.3,0.304,0.194,0.705,0.746,0.795,0.975,0.981
correl0.5,0.315,0.236,0.853,0.888,0.929,0.995,0.993


In [36]:
display(table_test_lin0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,,,
correl0.01,-0.010,-0.006,0.044,0.019,0.042,0.218,0.203
correl0.03,0.025,0.002,0.142,0.084,0.140,0.106,0.102
correl0.05,0.030,0.009,0.207,0.201,0.259,0.429,0.436
correl0.1,0.129,0.041,0.408,0.467,0.482,0.748,0.728
correl0.3,0.315,0.187,0.809,0.856,0.842,0.976,0.981
correl0.5,0.313,0.245,0.916,0.932,0.941,0.995,0.994


In [37]:
# Saving the data

table_train_lin0.to_csv('table_train_lin.csv', index=True)
table_test_lin0.to_csv('table_test_lin.csv', index=True)

In [38]:
# Later (even in a new notebook/session)
# table_train_cond = pd.read_csv('my_dataframe.csv', index_col=0)
# table_train_cond.head()

## Running on conditional effect

In [39]:
dic_models, dic_models_rollfalse = get_models()

table_train_cond0, table_test_cond0 = run_models('d_cond')

Running → correl0.01 | MLP_global


100%|██████████| 40/40 [00:06<00:00,  6.01it/s]


Running → correl0.01 | MLP_2D


100%|██████████| 40/40 [00:07<00:00,  5.22it/s]


Running → correl0.01 | trans_1D_T4


...

100%|██████████| 20/20 [00:07<00:00,  2.70it/s]


Running → correl0.5 | trans_2D_TCTC


100%|██████████| 20/20 [00:11<00:00,  1.71it/s]


Running → correl0.5 | trans_2D_TCTCTCTC


100%|██████████| 20/20 [00:21<00:00,  1.07s/it]


In [40]:
display(table_train_cond0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,,,
correl0.01,-0.004,0.015,0.028,0.028,0.031,0.070,0.082
correl0.03,0.011,0.023,0.092,0.089,0.092,0.109,0.101
correl0.05,0.013,0.035,0.146,0.143,0.138,0.288,0.323
correl0.1,0.027,0.072,0.288,0.273,0.271,0.471,0.527
correl0.3,0.072,0.177,0.679,0.655,0.645,0.796,0.784
correl0.5,0.094,0.221,0.832,0.827,0.801,0.916,0.905


In [41]:
display(table_test_cond0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,,,
correl0.01,0.014,0.005,0.008,0.004,0.011,0.056,0.077
correl0.03,0.004,-0.004,-0.014,-0.021,0.008,0.102,0.107
correl0.05,-0.005,-0.011,-0.003,0.006,0.009,0.253,0.295
correl0.1,-0.003,0.000,0.017,0.020,0.041,0.450,0.505
correl0.3,0.010,-0.005,0.027,0.063,0.072,0.729,0.717
correl0.5,-0.010,-0.006,0.010,0.476,0.138,0.883,0.861


In [42]:
# Saving the data

#table_train_cond0.to_csv('table_train_cond.csv', index=True)
#table_test_cond0.to_csv('table_test_cond.csv', index=True)

In [43]:
# Later (even in a new notebook/session)
# table_train_cond = pd.read_csv('my_dataframe.csv', index_col=0)
# table_train_cond.head()

## Running on shift effect

In [44]:
dic_models, dic_models_rollfalse = get_models()

table_train_shift0, table_test_shift0 = run_models('d_shift')

Running → correl0.01 | MLP_global


100%|██████████| 40/40 [00:06<00:00,  6.14it/s]


Running → correl0.01 | MLP_2D


100%|██████████| 40/40 [00:07<00:00,  5.56it/s]


Running → correl0.01 | trans_1D_T4


...

100%|██████████| 20/20 [00:07<00:00,  2.79it/s]


Running → correl0.5 | trans_2D_TCTC


100%|██████████| 20/20 [00:11<00:00,  1.75it/s]


Running → correl0.5 | trans_2D_TCTCTCTC


100%|██████████| 20/20 [00:21<00:00,  1.07s/it]


In [45]:
display(table_train_shift0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,,,
correl0.01,0.001,0.008,0.032,0.033,0.031,0.181,0.207
correl0.03,0.031,0.028,0.095,0.092,0.094,0.117,0.106
correl0.05,0.031,0.026,0.155,0.168,0.168,0.501,0.542
correl0.1,0.157,0.086,0.301,0.328,0.335,0.668,0.649
correl0.3,0.308,0.197,0.702,0.750,0.776,0.971,0.989
correl0.5,0.314,0.234,0.852,0.892,0.921,0.995,0.995


In [46]:
display(table_test_shift0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,,,
correl0.01,0.006,0.014,0.047,0.026,0.023,0.186,0.212
correl0.03,0.029,0.006,0.123,0.087,0.037,0.114,0.106
correl0.05,0.007,-0.001,0.220,0.206,0.120,0.489,0.548
correl0.1,0.130,0.023,0.404,0.456,0.309,0.673,0.667
correl0.3,0.300,0.184,0.814,0.858,0.804,0.972,0.988
correl0.5,0.317,0.249,0.914,0.931,0.923,0.995,0.995


In [47]:
# Saving the data

table_train_shift0.to_csv('table_train_shift.csv', index=True)
table_test_shift0.to_csv('table_test_shift.csv', index=True)

In [48]:
# Later (even in a new notebook/session)
# table_train_cond = pd.read_csv('my_dataframe.csv', index_col=0)
# table_train_cond.head()

## Running on the cs effect

In [49]:
dic_models, dic_models_rollfalse = get_models()

table_train_cs0, table_test_cs0 = run_models('d_cs')

Running → correl0.01 | MLP_global


100%|██████████| 40/40 [00:06<00:00,  6.36it/s]


Running → correl0.01 | MLP_2D


100%|██████████| 40/40 [00:07<00:00,  5.48it/s]


Running → correl0.01 | trans_1D_T4


...

100%|██████████| 20/20 [00:07<00:00,  2.72it/s]


Running → correl0.5 | trans_2D_TCTC


100%|██████████| 20/20 [00:11<00:00,  1.75it/s]


Running → correl0.5 | trans_2D_TCTCTCTC


100%|██████████| 20/20 [00:21<00:00,  1.06s/it]


In [50]:
display(table_train_cs0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,,,
correl0.01,0.009,0.010,0.032,0.031,0.037,0.058,0.062
correl0.03,0.045,0.033,0.093,0.088,0.098,0.109,0.102
correl0.05,0.064,0.053,0.168,0.176,0.208,0.454,0.596
correl0.1,0.163,0.087,0.328,0.353,0.394,0.804,0.649
correl0.3,0.300,0.188,0.701,0.751,0.788,0.972,0.962
correl0.5,0.314,0.238,0.856,0.894,0.932,0.990,0.990


In [51]:
display(table_test_cs0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,,,
correl0.01,0.022,0.015,0.051,0.026,0.055,0.054,0.069
correl0.03,0.027,0.003,0.149,0.092,0.132,0.141,0.140
correl0.05,0.061,0.023,0.239,0.183,0.271,0.456,0.588
correl0.1,0.145,0.049,0.439,0.494,0.500,0.813,0.672
correl0.3,0.307,0.177,0.811,0.858,0.840,0.972,0.962
correl0.5,0.307,0.243,0.917,0.933,0.941,0.990,0.990


In [52]:
# Saving the data

table_train_cs0.to_csv('table_train_cs.csv', index=True)
table_test_cs0.to_csv('table_test_cs.csv', index=True)

In [53]:
# Later (even in a new notebook/session)
# table_train_cond = pd.read_csv('my_dataframe.csv', index_col=0)
# table_train_cond.head()

## Running on the cs_shift

In [54]:
dic_models, dic_models_rollfalse = get_models()

table_train_csshift0, table_test_csshift0 = run_models('d_cs_shift')

Running → correl0.01 | MLP_global


100%|██████████| 40/40 [00:06<00:00,  6.18it/s]


Running → correl0.01 | MLP_2D


100%|██████████| 40/40 [00:07<00:00,  5.63it/s]


Running → correl0.01 | trans_1D_T4


...

100%|██████████| 20/20 [00:07<00:00,  2.78it/s]


Running → correl0.5 | trans_2D_TCTC


100%|██████████| 20/20 [00:11<00:00,  1.72it/s]


Running → correl0.5 | trans_2D_TCTCTCTC


100%|██████████| 20/20 [00:21<00:00,  1.07s/it]


In [55]:
display(table_train_csshift0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,,,
correl0.01,0.009,-0.002,0.027,0.028,0.027,0.002,0.002
correl0.03,0.021,0.031,0.104,0.102,0.106,0.097,0.094
correl0.05,0.057,0.046,0.158,0.166,0.162,0.051,0.057
correl0.1,0.161,0.081,0.315,0.343,0.348,0.174,0.119
correl0.3,0.308,0.192,0.699,0.743,0.778,0.973,0.974
correl0.5,0.312,0.238,0.854,0.891,0.923,0.989,0.987


In [56]:
display(table_test_csshift0.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

model,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC,trans_2D_TCTCTCTC
noise_level,,,,,,,
correl0.01,-0.000,0.016,0.054,0.031,0.025,0.007,0.014
correl0.03,0.013,0.014,0.158,0.100,0.035,0.009,0.007
correl0.05,0.031,0.010,0.227,0.205,0.112,0.017,0.018
correl0.1,0.148,0.032,0.429,0.488,0.313,0.073,0.052
correl0.3,0.293,0.164,0.810,0.853,0.806,0.973,0.974
correl0.5,0.313,0.253,0.914,0.932,0.924,0.989,0.987


In [57]:
# Saving the data

table_train_csshift0.to_csv('table_train_csshift.csv', index=True)
table_test_csshift0.to_csv('table_test_csshift.csv', index=True)

In [58]:
# Later (even in a new notebook/session)
# table_train_cond = pd.read_csv('my_dataframe.csv', index_col=0)
# table_train_cond.head()

# Function to create table for all effects

In [59]:
def run_models_all_effect(noise_level1, dic_models, dic_models_rollfalse):

    z = dic_data[noise_level1]["d_all"]
    z.get_dataloader(n_rolling=T1)

    lasso_full = ml_benchmarks.CustomBenchmarkRolling(LassoCV(alphas=np.linspace(start=1e-4, stop=1e-2, num=30), 
                                                            verbose=False, max_iter=2000, selection='random', n_jobs=5, cv=3))
    lasso_full.fit(z.train)
    boost_model = ml_benchmarks.CustomBenchmarkRolling(HistGradientBoostingRegressor(max_depth=6, max_iter=500, validation_fraction=0.3, n_iter_no_change=100))
    boost_model.fit(z.train)

    list_models = [lasso_full, boost_model]


    for model_key in dic_models.keys():                   
        print(f"Running → {noise_level1} | {model_key}")

        if model_key in ["trans_1D_T4", "trans_2D_TCTC", "trans_2D_TCTCTCTC"]:
            z.get_dataloader(n_rolling=T1, roll_y=True)
        else:
            z.get_dataloader(n_rolling=T1)

        if model_key in ["trans_2D_TCTC", "trans_2D_TCTCTCTC"]:
            lr=0.001/2
        else:
            lr=0.001

        epochs = 20
        if noise_level1 in ['correl0.01', 'correl0.03']:
            epochs = 40

        model = dic_models[model_key]

        # Model
        if noise_level1 == 'correl0.01' and model_key in ["trans_2D_TCTC", "trans_2D_TCTCTCTC"]:
            z.get_dataloader(n_rolling=T1)
            model = dic_models_rollfalse[model_key]
            epochs = 60
            lr=0.0001

        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        wrapper = torch_benchmarks.TorchWrapper(model, optimizer=optimizer, device=device)

        
        # Train
        wrapper.fit(z.train, test=z.test, epochs=epochs, plot=False)

        list_models.append(wrapper)

    comp = benchmark_comparison.Comparator(models=list_models, model_names=["lasso_full", "boosting"] + list(dic_models.keys()))

    corr_train = comp.correl(z, mode="train", return_values=True)
    corr_test  = comp.correl(z, mode="test",  return_values=True)

    return corr_train, corr_test

    

## Run on all noise levels

In [60]:
dic_models, dic_models_rollfalse = get_models()

all_effects_train_1, all_effects_test_1 = run_models_all_effect("correl0.01", dic_models, dic_models_rollfalse)

Running → correl0.01 | MLP_global


100%|██████████| 40/40 [00:06<00:00,  6.33it/s]


Running → correl0.01 | MLP_2D


100%|██████████| 40/40 [00:07<00:00,  5.47it/s]


Running → correl0.01 | trans_1D_T4


100%|██████████| 40/40 [00:14<00:00,  2.77it/s]


Running → correl0.01 | trans_2D_TCTC


100%|██████████| 60/60 [00:32<00:00,  1.85it/s]


Running → correl0.01 | trans_2D_TCTCTCTC


100%|██████████| 60/60 [01:02<00:00,  1.04s/it]


In [61]:
dic_models, dic_models_rollfalse = get_models()

all_effects_train_3, all_effects_test_3 = run_models_all_effect("correl0.03", dic_models, dic_models_rollfalse)

Running → correl0.03 | MLP_global


100%|██████████| 40/40 [00:06<00:00,  6.09it/s]


Running → correl0.03 | MLP_2D


100%|██████████| 40/40 [00:07<00:00,  5.57it/s]


Running → correl0.03 | trans_1D_T4


100%|██████████| 40/40 [00:14<00:00,  2.77it/s]


Running → correl0.03 | trans_2D_TCTC


100%|██████████| 40/40 [00:22<00:00,  1.75it/s]


Running → correl0.03 | trans_2D_TCTCTCTC


100%|██████████| 40/40 [00:43<00:00,  1.08s/it]


In [62]:
dic_models, dic_models_rollfalse = get_models()

all_effects_train_5, all_effects_test_5 = run_models_all_effect("correl0.05", dic_models, dic_models_rollfalse)

Running → correl0.05 | MLP_global


100%|██████████| 20/20 [00:03<00:00,  6.32it/s]


Running → correl0.05 | MLP_2D


100%|██████████| 20/20 [00:03<00:00,  5.61it/s]


Running → correl0.05 | trans_1D_T4


100%|██████████| 20/20 [00:07<00:00,  2.73it/s]


Running → correl0.05 | trans_2D_TCTC


100%|██████████| 20/20 [00:11<00:00,  1.75it/s]


Running → correl0.05 | trans_2D_TCTCTCTC


100%|██████████| 20/20 [00:21<00:00,  1.07s/it]


In [63]:
dic_models, dic_models_rollfalse = get_models()

all_effects_train_10, all_effects_test_10 = run_models_all_effect("correl0.1", dic_models, dic_models_rollfalse)

Running → correl0.1 | MLP_global


100%|██████████| 20/20 [00:03<00:00,  5.94it/s]


Running → correl0.1 | MLP_2D


100%|██████████| 20/20 [00:03<00:00,  5.58it/s]


Running → correl0.1 | trans_1D_T4


100%|██████████| 20/20 [00:07<00:00,  2.80it/s]


Running → correl0.1 | trans_2D_TCTC


100%|██████████| 20/20 [00:11<00:00,  1.75it/s]


Running → correl0.1 | trans_2D_TCTCTCTC


100%|██████████| 20/20 [00:21<00:00,  1.07s/it]


In [64]:
dic_models, dic_models_rollfalse = get_models()

all_effects_train_30, all_effects_test_30 = run_models_all_effect("correl0.3", dic_models, dic_models_rollfalse)

Running → correl0.3 | MLP_global


100%|██████████| 20/20 [00:03<00:00,  6.36it/s]


Running → correl0.3 | MLP_2D


100%|██████████| 20/20 [00:03<00:00,  5.64it/s]


Running → correl0.3 | trans_1D_T4


100%|██████████| 20/20 [00:07<00:00,  2.74it/s]


Running → correl0.3 | trans_2D_TCTC


100%|██████████| 20/20 [00:11<00:00,  1.75it/s]


Running → correl0.3 | trans_2D_TCTCTCTC


100%|██████████| 20/20 [00:21<00:00,  1.07s/it]


In [65]:
dic_models, dic_models_rollfalse = get_models()

all_effects_train_50, all_effects_test_50 = run_models_all_effect("correl0.5", dic_models, dic_models_rollfalse)

Running → correl0.5 | MLP_global


100%|██████████| 20/20 [00:03<00:00,  6.26it/s]


Running → correl0.5 | MLP_2D


100%|██████████| 20/20 [00:03<00:00,  5.29it/s]


Running → correl0.5 | trans_1D_T4


100%|██████████| 20/20 [00:07<00:00,  2.81it/s]


Running → correl0.5 | trans_2D_TCTC


100%|██████████| 20/20 [00:11<00:00,  1.75it/s]


Running → correl0.5 | trans_2D_TCTCTCTC


100%|██████████| 20/20 [00:21<00:00,  1.08s/it]


In [66]:
display(all_effects_train_1.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.035,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.018,0.450,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.016,0.447,0.011,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.010,0.446,-0.002,-0.004,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.017,0.440,-0.007,-0.005,-0.006,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.016,0.455,0.000,0.002,0.012,0.003,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.107,-0.008,0.000,-0.013,-0.009,0.009,-0.005,nan,nan,nan,nan,nan,nan
boosting,0.261,0.004,0.005,-0.009,-0.003,0.009,0.006,0.294,nan,nan,nan,nan,nan
MLP_global,0.989,0.034,0.019,0.015,0.010,0.017,0.016,0.108,0.263,nan,nan,nan,nan
MLP_2D,0.977,0.035,0.018,0.016,0.012,0.016,0.016,0.104,0.252,0.967,nan,nan,nan


In [67]:
display(all_effects_test_1.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.026,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.012,0.451,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.016,0.459,0.013,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.001,0.442,-0.006,0.008,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.018,0.441,-0.003,-0.001,0.000,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.010,0.444,0.010,-0.002,-0.011,-0.008,nan,nan,nan,nan,nan,nan,nan
lasso_full,-0.002,-0.009,0.001,-0.008,-0.001,-0.002,-0.011,nan,nan,nan,nan,nan,nan
boosting,-0.014,0.005,0.001,0.017,0.008,-0.011,-0.004,0.127,nan,nan,nan,nan,nan
MLP_global,0.010,0.028,0.021,-0.000,0.009,0.017,0.016,0.071,0.025,nan,nan,nan,nan
MLP_2D,-0.002,0.027,0.015,0.005,0.011,0.026,0.005,0.033,0.011,0.299,nan,nan,nan


In [68]:
display(all_effects_train_3.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.092,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.045,0.445,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.036,0.448,-0.007,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.035,0.449,0.005,0.005,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.043,0.445,-0.010,0.007,-0.006,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.047,0.451,0.005,-0.002,0.001,0.006,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.108,0.021,0.009,0.012,0.005,0.010,0.011,nan,nan,nan,nan,nan,nan
boosting,0.254,0.026,0.012,0.005,0.012,0.018,0.010,0.304,nan,nan,nan,nan,nan
MLP_global,0.989,0.094,0.045,0.037,0.036,0.045,0.049,0.108,0.249,nan,nan,nan,nan
MLP_2D,0.977,0.089,0.044,0.036,0.032,0.044,0.044,0.106,0.249,0.967,nan,nan,nan


In [69]:
display(all_effects_test_3.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.090,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.025,0.456,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.038,0.449,-0.001,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.048,0.450,0.012,0.003,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.044,0.454,0.019,0.006,-0.005,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.048,0.445,-0.000,-0.004,0.004,0.006,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.012,0.004,0.003,-0.014,0.008,0.006,0.007,nan,nan,nan,nan,nan,nan
boosting,-0.002,0.005,0.002,0.003,0.016,-0.013,0.002,0.114,nan,nan,nan,nan,nan
MLP_global,0.012,0.109,0.056,-0.003,0.056,0.068,0.068,0.078,0.024,nan,nan,nan,nan
MLP_2D,0.003,0.055,0.037,-0.007,0.027,0.031,0.036,0.045,0.013,0.324,nan,nan,nan


In [70]:
display(all_effects_train_5.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.158,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.069,0.454,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.073,0.452,0.006,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.073,0.451,0.009,-0.007,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.071,0.456,0.010,0.014,0.005,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.072,0.456,0.006,0.015,0.012,0.007,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.109,0.058,0.018,0.008,0.032,0.049,0.024,nan,nan,nan,nan,nan,nan
boosting,0.256,0.049,0.023,0.021,0.025,0.025,0.018,0.294,nan,nan,nan,nan,nan
MLP_global,0.983,0.162,0.071,0.072,0.077,0.073,0.075,0.109,0.250,nan,nan,nan,nan
MLP_2D,0.938,0.165,0.073,0.069,0.077,0.076,0.079,0.105,0.235,0.925,nan,nan,nan


In [71]:
display(all_effects_test_5.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.159,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.063,0.447,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.075,0.446,-0.004,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.066,0.465,0.014,0.011,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.068,0.455,-0.004,-0.002,0.026,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.087,0.451,0.001,0.005,0.008,0.007,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.003,0.038,0.010,-0.001,0.013,0.044,0.018,nan,nan,nan,nan,nan,nan
boosting,0.012,0.017,0.004,-0.004,0.014,0.011,0.014,0.188,nan,nan,nan,nan,nan
MLP_global,0.030,0.171,0.088,0.002,0.097,0.096,0.104,0.097,0.042,nan,nan,nan,nan
MLP_2D,0.031,0.134,0.067,-0.002,0.079,0.083,0.077,0.041,0.027,0.336,nan,nan,nan


In [72]:
display(all_effects_train_10.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.291,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.132,0.464,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.141,0.465,0.017,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.137,0.465,0.017,0.021,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.132,0.459,0.017,0.013,0.016,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.133,0.468,0.024,0.027,0.023,0.020,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.133,0.144,0.076,0.019,0.063,0.076,0.101,nan,nan,nan,nan,nan,nan
boosting,0.267,0.089,0.043,0.037,0.042,0.042,0.041,0.370,nan,nan,nan,nan,nan
MLP_global,0.984,0.298,0.137,0.138,0.143,0.135,0.140,0.136,0.263,nan,nan,nan,nan
MLP_2D,0.942,0.308,0.148,0.133,0.147,0.142,0.146,0.138,0.256,0.930,nan,nan,nan


In [73]:
display(all_effects_test_10.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.285,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.125,0.462,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.120,0.453,0.007,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.143,0.455,0.005,0.008,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.132,0.451,0.018,0.001,0.017,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.133,0.473,0.026,0.020,0.017,0.014,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.027,0.113,0.075,-0.007,0.048,0.067,0.076,nan,nan,nan,nan,nan,nan
boosting,0.013,0.045,0.032,0.000,0.012,0.024,0.035,0.224,nan,nan,nan,nan,nan
MLP_global,0.087,0.313,0.165,-0.000,0.176,0.187,0.190,0.127,0.042,nan,nan,nan,nan
MLP_2D,0.098,0.323,0.189,0.004,0.178,0.191,0.180,0.094,0.037,0.395,nan,nan,nan


In [74]:
display(all_effects_train_30.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.691,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.392,0.568,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.394,0.573,0.153,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.396,0.573,0.158,0.168,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.395,0.567,0.157,0.155,0.154,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.386,0.560,0.147,0.152,0.143,0.151,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.219,0.293,0.195,0.090,0.189,0.187,0.172,nan,nan,nan,nan,nan,nan
boosting,0.270,0.201,0.119,0.107,0.118,0.117,0.109,0.624,nan,nan,nan,nan,nan
MLP_global,0.987,0.705,0.403,0.391,0.407,0.406,0.396,0.226,0.268,nan,nan,nan,nan
MLP_2D,0.956,0.731,0.423,0.383,0.430,0.424,0.416,0.239,0.265,0.948,nan,nan,nan


In [75]:
display(all_effects_test_30.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.692,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.387,0.563,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.385,0.565,0.149,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.400,0.574,0.142,0.154,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.406,0.578,0.166,0.155,0.167,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.388,0.564,0.146,0.153,0.158,0.153,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.195,0.286,0.194,0.084,0.187,0.183,0.164,nan,nan,nan,nan,nan,nan
boosting,0.114,0.178,0.119,0.053,0.119,0.121,0.093,0.639,nan,nan,nan,nan,nan
MLP_global,0.518,0.742,0.473,0.197,0.483,0.488,0.465,0.253,0.162,nan,nan,nan,nan
MLP_2D,0.544,0.782,0.489,0.236,0.509,0.498,0.489,0.269,0.169,0.733,nan,nan,nan


In [76]:
display(all_effects_train_50.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.844,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.575,0.682,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.577,0.683,0.327,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.581,0.688,0.340,0.341,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.580,0.687,0.342,0.336,0.336,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.580,0.686,0.331,0.335,0.341,0.340,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.259,0.301,0.217,0.151,0.224,0.218,0.222,nan,nan,nan,nan,nan,nan
boosting,0.265,0.234,0.159,0.153,0.157,0.164,0.167,0.738,nan,nan,nan,nan,nan
MLP_global,0.989,0.852,0.583,0.572,0.587,0.587,0.588,0.264,0.265,nan,nan,nan,nan
MLP_2D,0.965,0.879,0.604,0.564,0.613,0.613,0.617,0.271,0.257,0.961,nan,nan,nan


In [77]:
display(all_effects_test_50.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

,true,optimal,linear,conditional,shift,cs,cs_shift,lasso_full,boosting,MLP_global,MLP_2D,trans_1D_T4,trans_2D_TCTC
optimal,0.844,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
linear,0.575,0.683,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
conditional,0.564,0.676,0.320,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
shift,0.573,0.683,0.335,0.324,nan,nan,nan,nan,nan,nan,nan,nan,nan
cs,0.580,0.682,0.332,0.331,0.336,nan,nan,nan,nan,nan,nan,nan,nan
cs_shift,0.584,0.684,0.338,0.331,0.331,0.330,nan,nan,nan,nan,nan,nan,nan
lasso_full,0.258,0.302,0.226,0.152,0.235,0.208,0.208,nan,nan,nan,nan,nan,nan
boosting,0.201,0.236,0.175,0.120,0.174,0.170,0.166,0.780,nan,nan,nan,nan,nan
MLP_global,0.733,0.867,0.636,0.414,0.635,0.634,0.635,0.287,0.230,nan,nan,nan,nan
MLP_2D,0.755,0.891,0.646,0.442,0.651,0.644,0.653,0.290,0.235,0.864,nan,nan,nan


In [78]:
# Let's also save all the tables above.
all_effects_train_1.to_csv('all_effects_train_1.csv', index=True)
all_effects_test_1.to_csv('all_effects_test_1.csv', index=True)

all_effects_train_3.to_csv('all_effects_train_3.csv', index=True)
all_effects_test_3.to_csv('all_effects_test_3.csv', index=True)

all_effects_train_5.to_csv('all_effects_train_5.csv', index=True)
all_effects_test_5.to_csv('all_effects_test_5.csv', index=True)

all_effects_train_10.to_csv('all_effects_train_10.csv', index=True)
all_effects_test_10.to_csv('all_effects_test_10.csv', index=True)

all_effects_train_30.to_csv('all_effects_train_30.csv', index=True)
all_effects_test_30.to_csv('all_effects_test_30.csv', index=True)

all_effects_train_50.to_csv('all_effects_train_50.csv', index=True)
all_effects_test_50.to_csv('all_effects_test_50.csv', index=True)

# Testing sparsity

In [97]:
# To test sparsity let's write a function that runs one model on all the effects and noise levels.
# Then we can run on the model with and without sparsity.

In [98]:
def keep_topk_per_row(x, k=3):
    vals, idx = torch.topk(x, k=k, dim=-1, largest=True)
    out = torch.zeros_like(x)
    out.scatter_(-1, idx, 1)
    return out


In [99]:
effects

['d_lin', 'd_cond', 'd_shift', 'd_cs', 'd_cs_shift', 'd_all']

In [100]:
def test_sparsity_all_correl_effets(sparsity=False):

    # Storage
    records_train = []
    records_test  = []

    for noise_level in dic_data.keys():          
        for effect in ['d_lin', 'd_cond', 'd_shift', 'd_cs', 'd_cs_shift', 'd_all']:                   
            print(f"Running → {noise_level} | {effect}")

            z = dic_data[noise_level][effect]
            z.get_dataloader(n_rolling=T1, roll_y=True)
            
            lr=0.001/2
            roll_y=True
            
            epochs = 20
            if noise_level in ['correl0.01', 'correl0.03']:
                epochs = 40


            # Model
            if noise_level == 'correl0.01':
                z.get_dataloader(n_rolling=T1)
                roll_y=False
                epochs = 60
                lr=0.0001

            # Model
            if sparsity:
                model = models.CustomBiDimensionalTransformer(N1, F1, T1, mask=mask, layers='TCTC',
                                                    dropout=0.05, d_model=128, dim_feedforward=128*4, sparsify=keep_topk_per_row, 
                                                                               roll_y=roll_y).to(device)
            else:
                model = models.CustomBiDimensionalTransformer(N1, F1, T1, mask=mask, layers='TCTC',
                                                        dropout=0.05, d_model=128, dim_feedforward=128*4, sparsify=None, 
                                                                                roll_y=roll_y).to(device)
            
            optimizer = torch.optim.Adam(model.parameters(), lr=lr)
            wrapper = torch_benchmarks.TorchWrapper(model, optimizer=optimizer, device=device)

            # Train
            wrapper.fit(z.train, test=z.test, epochs=epochs, plot=False)

            # Compare
            comp = benchmark_comparison.Comparator(models=[wrapper], model_names=["model1"])

            corr_train = comp.correl(z, mode="train", return_values=True)
            corr_test  = comp.correl(z, mode="test",  return_values=True)

            train_corr = corr_train.loc["model1", "optimal"]
            test_corr  = corr_test.loc["model1", "optimal"]

            # Save results
            records_train.append({
                "noise_level": noise_level,
                "effect": effect,
                "train_corr_optimal": train_corr
            })
            records_test.append({
                "noise_level": noise_level,
                "effect": effect,
                "test_corr_optimal": test_corr
            })

    # ————————————————————————————————————————
    # 3. Convert to nice DataFrames
    # ————————————————————————————————————————
    df_train = pd.DataFrame(records_train)
    df_test  = pd.DataFrame(records_test)

    # Pivot: rows = noise level, columns = effect type
    train_pivot = df_train.pivot(index="noise_level", columns="effect", values="train_corr_optimal")
    test_pivot  = df_test.pivot(index="noise_level", columns="effect", values="test_corr_optimal")    

    # Sort columns in logical order
    col_order = ["d_lin", "d_cond", "d_shift", "d_cs", "d_cs_shift", "d_all"]
    train_model = train_pivot[col_order]
    test_model  = test_pivot[col_order]

    return train_model, test_model

In [101]:
table_train_no_sparsity, table_test_no_sparsity = test_sparsity_all_correl_effets()

Running → correl0.01 | d_lin


  0%|          | 0/60 [00:00<?, ?it/s]

100%|██████████| 60/60 [00:32<00:00,  1.83it/s]


Running → correl0.01 | d_cond


100%|██████████| 60/60 [00:32<00:00,  1.83it/s]


...

100%|██████████| 20/20 [00:11<00:00,  1.69it/s]


Running → correl0.5 | d_cs_shift


100%|██████████| 20/20 [00:11<00:00,  1.74it/s]


Running → correl0.5 | d_all


100%|██████████| 20/20 [00:11<00:00,  1.73it/s]


In [102]:
table_train_with_sparsity, table_test_with_sparsity = test_sparsity_all_correl_effets(sparsity=True)

Running → correl0.01 | d_lin


  0%|          | 0/60 [00:00<?, ?it/s]

100%|██████████| 60/60 [00:33<00:00,  1.81it/s]


Running → correl0.01 | d_cond


100%|██████████| 60/60 [00:33<00:00,  1.81it/s]


...

100%|██████████| 20/20 [00:11<00:00,  1.68it/s]


Running → correl0.5 | d_cs_shift


100%|██████████| 20/20 [00:11<00:00,  1.71it/s]


Running → correl0.5 | d_all


100%|██████████| 20/20 [00:11<00:00,  1.71it/s]


In [103]:
display(table_train_no_sparsity.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

effect,d_lin,d_cond,d_shift,d_cs,d_cs_shift,d_all
noise_level,,,,,,
correl0.01,0.132,0.038,0.203,0.062,0.004,0.093
correl0.03,0.113,0.112,0.120,0.103,0.092,0.106
correl0.05,0.485,0.293,0.648,0.474,0.036,0.400
correl0.1,0.730,0.439,0.766,0.801,0.366,0.546
correl0.3,0.976,0.811,0.977,0.972,0.974,0.940
correl0.5,0.996,0.915,0.994,0.990,0.989,0.979


In [104]:
display(table_train_with_sparsity.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

effect,d_lin,d_cond,d_shift,d_cs,d_cs_shift,d_all
noise_level,,,,,,
correl0.01,0.130,0.054,0.164,0.058,0.010,0.106
correl0.03,0.121,0.109,0.110,0.114,0.096,0.101
correl0.05,0.478,0.296,0.552,0.445,0.091,0.318
correl0.1,0.766,0.439,0.807,0.720,0.452,0.584
correl0.3,0.971,0.819,0.963,0.967,0.973,0.951
correl0.5,0.996,0.919,0.994,0.991,0.989,0.979


In [105]:
display(table_test_no_sparsity.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

effect,d_lin,d_cond,d_shift,d_cs,d_cs_shift,d_all
noise_level,,,,,,
correl0.01,0.138,0.037,0.209,0.065,-0.011,0.090
correl0.03,0.126,0.082,0.114,0.120,0.006,0.104
correl0.05,0.498,0.263,0.647,0.474,0.014,0.382
correl0.1,0.731,0.406,0.771,0.806,0.250,0.533
correl0.3,0.977,0.750,0.978,0.972,0.974,0.937
correl0.5,0.996,0.882,0.994,0.990,0.989,0.977


In [106]:
display(table_test_with_sparsity.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format(precision=3))

effect,d_lin,d_cond,d_shift,d_cs,d_cs_shift,d_all
noise_level,,,,,,
correl0.01,0.147,0.049,0.171,0.062,-0.001,0.101
correl0.03,0.132,0.092,0.108,0.130,0.011,0.076
correl0.05,0.498,0.270,0.541,0.441,0.030,0.302
correl0.1,0.772,0.406,0.814,0.733,0.365,0.556
correl0.3,0.972,0.762,0.964,0.967,0.973,0.950
correl0.5,0.996,0.889,0.994,0.991,0.989,0.977


In [96]:
# To store the tables above

table_train_no_sparsity.to_csv('table_train_no_sparsity.csv', index=True)
table_test_no_sparsity.to_csv('table_test_no_sparsity.csv', index=True)

table_train_with_sparsity.to_csv('table_train_with_sparsity.csv', index=True)
table_test_with_sparsity.to_csv('table_test_with_sparsity.csv', index=True)